In [1]:
from openpyxl import load_workbook
from collections import namedtuple, defaultdict
import pandas as pd
import re, os, shutil
from pathlib import Path
from pprint import pprint
import numpy as np
from fnmatch import fnmatch

In [3]:
# pohjapolku, haetaan kansiot sen perusteella löytyykö laskentapohjaa

p = Path(r'C:\Users\Heikki\OneDrive - HJT-RAK Oy\001 TOIMISTOURAKAT')

files = [str(x) for x in p.rglob('*laskentapohj*')]

In [4]:
# edellisen kohdan perusteella haetaan kansiot jotka kiinnostaa

nones = ['005', 'POHJAKANSIO']
paths = defaultdict(list)

for file in files:
    file = str(file)
    m = re.search(r'(\d{4}[\w\s][^\\]+)', file)
    file = file[:m.end()] if m else None
    if file is None or any(x in str(file) for x in nones):
        continue
    paths[m.group(0)] = Path(file)

paths = dict(paths)


In [ ]:
# luodaan projektikohtasesti dictit kiinnostaville projekteille

koonti = {}

for project, path in paths.items():
    selostukset = list(path.rglob('*selostu*.pdf'))
    laskenta_excelit = list(path.rglob('*laskentapoh*.xlsx'))
    piirustukset = list(path.rglob('*laskenta*.pdf')) + list(path.rglob('*massaluet*.pdf'))
    koonti[project] = {
        'selostukset': selostukset,
        'laskenta_excelit': laskenta_excelit,
        'piirustukset': piirustukset
    } 



In [6]:
print(f'data_parser_input, projektien määrä: {len(list(Path(r'C:\Users\Heikki\data_parser_input').iterdir()))}')
print(f'oma projektimäärä: {len(list(koonti.keys()))}')

data_parser_input, projektien määrä: 122
oma projektimäärä: 166


In [ ]:
# luodaan projektikansiot ja kopioidaan laskentamatsku
# ÄLÄ AJA TOISTA KERTAA, kelvottomat projektit pitää joka tapauksessa tarkastaa ja poistaa käsin
'''
for projekti, listat in koonti.items():
    os.makedirs(f'./projektit/{projekti}/', exist_ok=True)
    for lista, polku in listat.items():
        os.makedirs(f'./projektit/{projekti}/{lista}/', exist_ok=True)
        for p in polku:
            shutil.copy(p, f'./projektit/{projekti}/{lista}/')
'''

In [ ]:
# kato karkeasti käsin käpi kelvottomat projektit

path = Path('./projektit/')
paths = [p for p in path.iterdir()]

dirs_to_remove = []

for path in paths:
    p = str(path)
    m = re.search(r'\d{4}\s\w[^\\]+', p)
    m = m.group(0) if  m else None
    selostukset = list(path.rglob('*selostus*.pdf'))
    laskentasarjat = list(path.rglob('*laskenta*.xlsx'))
    piirustukset = list(path.rglob('*laskenta*.pdf'))
    if len(laskentasarjat) == 0 or len(piirustukset) == 0:
        print(f'tyhjä polku: {m}')
        print()
        continue
    # print(f'path: {m}')
    # print(f'selostukset: {selostukset}')
    # print(f'laskentasarjat: {laskentasarjat}')
    # print(f'piirustukset: {piirustukset}')
    # print()

In [2]:
file_name = r"001 laskentapohja, 2024.xlsx"
wb = load_workbook(filename=file_name, data_only=True)

In [ ]:
wb.sheetnames

In [10]:
# [x[:3] for x in wb['Purkutyöt'].iter_rows(values_only=True) if x[:3].count(None) < 2]
# [x for x in wb['Materiaalihinnoittelut'].iter_rows(values_only=True) if isinstance(x[-1], int) and x[-1] > 0]
# [x for x in wb['Tuntihinnoittelut'].iter_rows(values_only=True) if x.count(None) < 2]

In [ ]:
# erillishankinnat

# line = namedtuple('item', ['työ', 'hinta'])

# lines = [row[1:3] for row in wb['Erillishankinnat'].iter_rows(
#     values_only=True
#     )
#    if str(row[1:3][0]) not in ('Kate-%', 'Kate-osuus')
#    and row[1:3].count(None) < 1
#    ][1:]

# lines = [line for line in lines if line[1] > 30]

# item = [x[0] for x in lines[::2]]
# hinta = [x[1] for x in lines[1::2]]

# lines = [x for x in zip(item, hinta)]
# lines = [line(*x) for x in lines]

# df = pd.DataFrame(lines, index=[file_name] * len(lines))
# df_e = pd.DataFrame([{'selite': 'erillishankinnat', 'materiaalit': df['hinta'].sum(), 'työt': 0}])
# df_e

In [ ]:
p = Path("./projektit")
files = [a for a in p.rglob('*laskenta*.xlsx')]

for i, file in enumerate(files):
    file_name = file
    m = re.search(r'\d{4}[\w\s.-]+[^\\,]', str(file))
    m = m.group()
    print(i, m, file_name)

In [ ]:
dfs_X = []

for j, file in enumerate(files):
    try:
        file_name = file
        m = re.search(r'\d{4}[\w\s.-]+[^\\,]', str(file))
        m = m.group()
        wb = load_workbook(filename=file_name, data_only=True)

        # erittelysivu

        line1 = namedtuple('item', ['selite', 'materiaalit', 'työt'])
        line2 = namedtuple('item', ['Erillishankinnat', 'Hinta', 
                                    'Toimittaja_urakoitsija', 'ignore1', 
                                    'Toimitus', 'ignore2', 'ignore3'])

        erittelyt_cols_left = []
        erittelyt_cols_right = []
        erillishankinnat_cols = []
        kokonaishinnat = []

        temp = []

        for i, row in enumerate(wb['Eritelysivu'].iter_rows(values_only=True)):
            if i < 77:
                erittelyt_cols_left.append(line1(*row[:3]))
                erittelyt_cols_right.append(line1(*row[4:7]))
            if 77 <= i < 97:
                erittelyt_cols_left.append(line1(*row[:3]))
            if 97 <= i < 114:
                erillishankinnat_cols.append(line2(*row))
            if i > 114:
                if row.count(None) < 5:
                    if len(temp) == 0:
                        ntuple_names = list(row)
                        ntuple_names = [f'id_{i}' if x is None else x for i, x in enumerate(ntuple_names)]
                        ntuple_names = [x.replace('-', '_') for x in ntuple_names]
                        ntuple = namedtuple('item', ntuple_names)
                        temp.append(ntuple)
                        continue
                    if len(temp) != 0:
                        out = temp.pop()
                        out = out(*row)
                        kokonaishinnat.append(out)


        df_erillishankinnat = pd.DataFrame(erillishankinnat_cols).iloc[:, [0, 1,4]]

        df_left = pd.DataFrame(erittelyt_cols_left)
        df_right = pd.DataFrame(erittelyt_cols_right)
        df_right.columns = df_left.columns
        df_erittelyt = pd.concat([df_left, df_right])

        df = df_erittelyt[df_erittelyt['selite'].astype(str).str.contains('tuntihin|välis|ovet|hinta|tunnit', case=False)]
        df[['materiaalit', 'työt']] = df[['materiaalit', 'työt']].apply(pd.to_numeric, errors='coerce')
        df = df.dropna(thresh=2)

        # korjaa laatoituksen otsikko
        mask = df["selite"].eq("Hinta, lattiat")

        if mask.sum() >= 2:
            second_idx = df.index[mask][1]
            df.loc[second_idx, "selite"] = "Hinta, laatoitukset"

        #df = pd.concat([df, df_e])

        d = df.copy()

        # laatoitus-kohdan kirjoitusvirheen korjaus
        mask = d["selite"].eq("Hinta, materiaalit")
        d.loc[mask, "materiaalit"] = d.loc[mask, "työt"]
        d.loc[mask, "työt"] = 0

        # category mapping

        def get_category(s):
            s = str(s).lower()

            if "tuntihinnoittelut" in s or "materiaalit" in s:
                return "työmaan yleiskulut"
            if "purkutyöt" in s:
                return "purkutyöt"
            if "maalaukset" in s or "maalaus" in s:
                return "maalaukset"
            if "lattiat" in s:
                return "lattian_asennukset"
            if "laatoit" in s:
                return "laatoitukset"
            if "väliseinät" in s:
                return "väliseinät"
            if "alakatot" in s:
                return "alakatot"
            if "ovet" in s or "listat" in s:
                return "ovet_ja_listoitukset"

            # return "erillishankinnat" # jätetään erillishankinnat pois

        d["category"] = d["selite"].map(get_category)

        # hour rows have hours in the "työt" column
        is_hours = d["selite"].str.contains("tunnit", case=False, na=False)

        d["tunnit"] = np.where(is_hours, d["työt"], 0)
        d.loc[is_hours, ["materiaalit", "työt"]] = 0

        target = (
            d.groupby("category", as_index=False)
            .agg({
                "materiaalit": "sum",
                "työt": "sum",
                "tunnit": "sum",
            })
        )

        target["yhteensä"] = target["materiaalit"] + target["työt"]
        target.index = [f'{m}_{j}'] * len(target.index)
        dfs_X.append(target)
    except Exception as e:
        print(f'file {m} error: {e}')

In [52]:
out = pd.concat(dfs_X)

In [53]:
out['project'] = out.index

In [54]:
out = out.pivot_table(
    index='project',
    columns='category',
    values='yhteensä',
    aggfunc='sum',
    fill_value=0
).reset_index()

In [ ]:
# pistetään exceliin talteen varmuuden vuoksi

out.to_excel('targets.xlsx', engine='openpyxl')